In [1]:
#! /usr/bin/env python3
# -*- coding: utf-8 -*-

#
# Modified by gli945 on 01/09/2025..
# Calculate the average query time of each algorithm for each dataset-query combination and draw the bar chart.
#

import argparse
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
def get_args():
    parser = argparse.ArgumentParser(description='Draw the average query time of each algorithm for each dataset-query combination.')
    parser.add_argument('--input_dir', type=str, default='results/csv-v1/overview', 
                        help='The path of the input directory that contains the query-time csv files.')
    parser.add_argument('--result_dir', type=str, default='results/fig-v2', help='The path to the output files.')
    return parser.parse_args()

kAlgorithmList = ['SymBi', 'RapidFlow', 'NewSP', 'GCSM_CPU', 'GAMMA*', 'QO-GAMMA*', 'GCSM-BU', 'QO-GAMMA*(CPU)', 'GAMMA*(CPU)']
query_shape_list = ['tree_6', 'sparse_6', 'dense_6']

dataset_list = ['lsbench_x1', 'netflow_', 'livejournal_30', 'amazon_6']

algorithm_list = ['GAMMA*', 'QO-GAMMA*', 'GAMMA*', 'QO-GAMMA*']

color_dict = {
    'SymBi': '#ffefd8',
    'NewSP': '#ffc876',
    'RapidFlow': '#ff8300',
    'GCSM_CPU': None,
    'GAMMA*': None,
    'QO-GAMMA*': None,
    'GCSM-BU': '#ff6514',
    'QO-GAMMA*(CPU)': None,
    'GAMMA*': '#ffa576'
}

hatch_dict = {
    'SymBi': '',
    'NewSP': '',
    'RapidFlow': '',
    'GCSM_CPU': '',
    'GAMMA*': 'x',
    'QO-GAMMA*': '/',
    'GCSM-BU': '\\',
    'QO-GAMMA*(CPU)': '',
    'GAMMA*(CPU)': ''
}

# color_dict = {
#     'SymBi': '#ffefd8',
#     'NewSP': '#ffc876',
#     'RapidFlow': '#ff8300',
#     'GCSM_CPU': None,
#     'GAMMA': None,
#     'CorrectGamma': '#ffa576',
#     'GCSM-BU': '#ff6514'
# }

# hatch_dict = {
#     'SymBi': '',
#     'NewSP': '',
#     'RapidFlow': '',
#     'GCSM_CPU': '',
#     'GAMMA': 'x',
#     'CorrectGamma': '/',
#     'GCSM-BU': '\\'
# }

abbreviation_dict = {
    'livejournal_30': 'lj',
    'amazon_6': 'az',
    'lsbench_x1': 'lb',
    'netflow_': 'nf'
}

fullname_dict = {
    'livejournal_30': 'LiveJournal',
    'amazon_6': 'Amazon',
    'lsbench_x1': 'LSBench',
    'netflow_': 'Netflow'
}

def find_algorithm_index(algorithm):
    for i in range(len(kAlgorithmList)):
        if kAlgorithmList[i].lower() == algorithm.lower():
            return i
    return -1

def load_query_time_for_all_algorithms(input_dir, dataset, query_shape):
    return np.loadtxt(f'{input_dir}/{dataset}_{query_shape}.csv', delimiter=',')

# Select the query time for queries where all algorithms within the time limit
def select_within_time_limit(query_time, time_limit, algorithm_ids):
    # np.max([1.0, 2.0, nan]) == nan, (nan < 'any_number') == false, so we can remove nan
    mask = np.max(query_time[:,algorithm_ids], axis=1) < time_limit
    # return mask
    return query_time[mask][:,algorithm_ids]

def truncate_query_time(query_time, time_limit):
    time_out_mask = np.logical_not(np.max(query_time, axis=1) < time_limit)
    query_time[time_out_mask] = time_limit
    return query_time

def calculate_average_query_time(masked_query_time, algorithm_id):
    return np.average(masked_query_time[:,algorithm_id])

def generate_x_positions(num_bars, x_ticks, bar_width=0.14, gap_size=0.04):
    # num_bars = len(algorithm_list)  # number of bars at each x tick
    left_most_bar_offset = - (num_bars/2 - 1/2) * bar_width - ((num_bars-1)/2 - 1/2) * gap_size
    offset_list = [left_most_bar_offset]
    for _ in range(1, num_bars):
        offset_list.append(offset_list[-1] + bar_width + gap_size)
    offset_array = np.expand_dims(np.array(offset_list), axis=1)  # shape: (num_bars, 1)
    x_ticks = np.repeat(np.expand_dims(x_ticks, axis=0), num_bars, axis=0)  # shape: (num_bars, num_x_ticks)
    x_positions = x_ticks + offset_array
    return x_positions


In [ ]:
# os.chdir(f"{os.getcwd()}/../../")

# result_dir = "./charts"
result_dir = "../../charts"
# os.makedirs(result_dir, exist_ok=True)

input_dir = "../../output/query_time"

font_size = 22
bar_width = 0.40
# gap_size = 0.01
gap_size = 0
show_bar_label = True

# algorithm_list_v1 = ['GAMMA*', 'QO-GAMMA*', 'GAMMA*(CPU)', 'QO-GAMMA*(CPU)']
algorithm_list_v1 = ['GAMMA*', 'QO-GAMMA*']

fig, axss = plt.subplots(1, 3, figsize=(16.8, 4.2), sharey=True)

subplot_titles = ['Tree', 'Sparse', 'Dense']

old_algorithm_list = algorithm_list_v1.copy()
old_algorithm_id_list = [find_algorithm_index(algorithm) for algorithm in old_algorithm_list]

# Select the query time for queries where all algorithms within the time limit
def get_mask_within_time_limit(query_time, time_limit, algorithm_ids):
    # np.max([1.0, 2.0, nan]) == nan, (nan < 'any_number') == false, so we can remove nan
    mask = np.max(query_time[:,algorithm_ids], axis=1) < time_limit
    return mask
    # return query_time[mask][:,algorithm_ids]


query_shape_list = ['tree_6', 'sparse_6', 'dense_6']

# for row_idx in range(axss.shape[0]):
for row_idx in range(1):
    # axs = axss[row_idx]
    axs = axss
    algorithm_list = old_algorithm_list[2*row_idx:2*row_idx+2]
    algorithm_id_list = [find_algorithm_index(algorithm) for algorithm in algorithm_list]
    for col_idx, query_name in enumerate(query_shape_list):

        cur_dataset_list = dataset_list
        
        query_times_for_datasets = []
        for dataset in cur_dataset_list:
            query_time = load_query_time_for_all_algorithms(input_dir, dataset, query_name)
            
            # query_time = select_within_time_limit(query_time, 1800000, algorithm_id_list)
            # query_time = select_within_time_limit(query_time, 1800000, old_algorithm_id_list)
            mask = get_mask_within_time_limit(query_time, 1800000, old_algorithm_id_list)
            
            # mask = mask & mask_2
            query_time = query_time[mask][:, old_algorithm_id_list]
            query_time = query_time[:, 2*row_idx:2*row_idx+2]
            query_times_for_datasets.append(query_time)

        average_query_times = []
        for algorithm_id_idx in range(len(algorithm_id_list)):
            cur_averages = []
            for query_time in query_times_for_datasets:
                average_query_time = np.average(query_time[:,algorithm_id_idx])  # scalar
                average_query_time = average_query_time / 1000
                average_query_time = np.around(average_query_time, decimals=3)
                cur_averages.append(average_query_time)
            average_query_times.append(cur_averages)
        
        x_positions = generate_x_positions(len(algorithm_list), np.arange(len(cur_dataset_list)), bar_width=bar_width, gap_size=gap_size)

        y_threshold = 2.0

        idx = 0
        for x, query_time in zip(x_positions, average_query_times):
            algorithm = algorithm_list[idx]
            y = []
            for time in query_time:
                if time < y_threshold:
                    y.append(time)
                else:
                    y.append(y_threshold)
            rects = axs[col_idx].bar(x, y, width=bar_width, edgecolor='black', label=algorithm, 
                            color=color_dict[algorithm], hatch=hatch_dict[algorithm])
            if show_bar_label:
                anno_ls = axs[col_idx].bar_label(rects, labels=query_time, fontsize=font_size-5, rotation=55, padding=0)
            idx += 1

        
        axs[col_idx].set_ylim(0.0, y_threshold)
        
        if row_idx == 0:
            axs[col_idx].set_title(subplot_titles[col_idx], fontsize=font_size, pad=52)
        
        axs[col_idx].set_xticks(list(range(len(cur_dataset_list))))
        axs[col_idx].set_xticklabels(
            [f"${abbreviation_dict[dataset]}$\n" for dataset in cur_dataset_list], 
            fontsize=font_size
        )

        axs[col_idx].tick_params('y', labelsize=font_size-3)

    if row_idx == 0:
        axs[0].set_ylabel(f'Query Time        \n  (in seconds)        ', fontsize=font_size-4, rotation=90)
    else:
        axs[0].set_ylabel(f'  CPU         \n Version        \n  Query Time        \n  (in seconds)        ', fontsize=font_size-4, rotation=0)


handles_1, labels_1 = axss[0].get_legend_handles_labels()

fig.legend(handles_1, labels_1, facecolor='white', framealpha=0, ncol=5, bbox_to_anchor=(0.5, 1), loc=9, fontsize=font_size)
fig.tight_layout(rect=(0,0,1,0.89))

fig.savefig(f'{result_dir}/1-QO-DO_comparison_plot.pdf')
plt.show()
plt.close()
